This section addes minimum to the main dataset yet requires API Search and deep scan of yml and their supporting files which does not worth it at this stage

In [11]:
# -*- coding: utf-8 -*-
import os
import re
import json
import pandas as pd
from pathlib import Path

try:
    import yaml  # pip install pyyaml
except ImportError:
    raise SystemExit("Please install pyyaml: pip install pyyaml")

# ========= CONFIG =========
MAIN_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.2_Total_Repo.csv"
ALL_CFG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
OUT_CSV = os.path.join(OUT_DIR, "3.3.1_Repo_CI_Instru_Test_Scan.csv")
os.makedirs(OUT_DIR, exist_ok=True)

CI_YAML_EXTS = {".yml", ".yaml"}
Search_Method_Name = "Phase 2 - Workflow & Script Trace Scan"  # <-- NEW

# ========= DETECTION REGEX =========
RE_CLOUD = re.compile(
    r"(?:\b(?:sudo\s+)?gcloud(?:\s+beta)?(?:\s+--quiet)?\s+firebase\s+test\s+android\s+run\b"
    r"|saucectl(?:\s+run)?\b"
    r"|browserstack\b)",
    re.IGNORECASE,
)
RE_TRIGGER = re.compile(
    r"(?:(?:\b(?:sudo\s+)?)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*\b(?:connectedAndroidTest|[\w:-]*managedDevice\w*AndroidTest|connectedcheck)\b"
    r"|flutter\s+(?:test|drive)\b.*\bintegration_test\b)",
    re.IGNORECASE,
)
RE_DEVICE = re.compile(
    r"(?:(?:\b)avdmanager\b|(?<!android-)emulator\b|android-wait-for-emulator\b"
    r"|reactivecircus/android-emulator-runner|hannesa2/action-android/emulator-run-cmd"
    r"|sdkmanager\s+[\"']system-images;android-\d+;[^\"']+[\"']|manageddevice\w*androidtest\b)",
    re.IGNORECASE,
)
RE_EXCLUDE = re.compile(r"(?:\bgradle(?:w|\.bat)?\s+.*\b(?:build|check|test)\b|\bjvm[-_ ]?tests?\b)", re.IGNORECASE)
RE_LOCAL_SCRIPT = re.compile(r"(^|\s)(?P<cmd>(?:bash|sh)\s+)?(?P<path>\.?/?(?:ci|scripts|tools|\.github/actions)/[^\s;|&]+)", re.IGNORECASE)
RE_MAKE = re.compile(r"\bmake\s+(?P<target>[\w:-]+)", re.IGNORECASE)
RE_FASTLANE = re.compile(r"\bfastlane\s+(?P<lane>[\w:_-]+)", re.IGNORECASE)
RE_LOCAL_ACTION_USES = re.compile(r"^\s*uses:\s*[\"']\s*\./\.github/actions/([^\"']+)", re.IGNORECASE)

# ========= HELPERS =========
COMMENT_LINE_RE = re.compile(r"(?m)^\s*(#|//|REM\b|::).*$")
def strip_comments(text: str) -> str:
    return COMMENT_LINE_RE.sub("", text or "")

def read_text(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        try:
            return path.read_text(encoding="latin1", errors="ignore")
        except Exception:
            return ""

def list_repo_files(repo_id: str) -> list[Path]:
    results = []
    repo_id_lower = repo_id.lower()
    for p in Path(ALL_CFG_DIR).rglob("*"):
        if p.is_file():
            name_lower = p.name.lower()
            path_lower = str(p).lower()
            if name_lower.startswith(repo_id_lower + "__") or repo_id_lower in path_lower:
                results.append(p)
    return results

def yaml_like(path: Path) -> bool:
    n = path.name.lower()
    return (path.suffix.lower() in CI_YAML_EXTS
            or n in {".travis.yml", "jenkinsfile", "azure-pipelines.yml", "bitrise.yml"})

def parse_yaml_runs_uses(yaml_text: str):
    lines = yaml_text.splitlines()
    runs, uses_vals = [], []
    try:
        data = yaml.safe_load(yaml_text)
    except Exception:
        data = None

    def collect_cmds(val):
        if isinstance(val, list):
            for x in val:
                if isinstance(x, str):
                    runs.append(x)
        elif isinstance(val, str):
            runs.append(val)

    def process_job(job_dict):
        if not isinstance(job_dict, dict):
            return
        steps = job_dict.get("steps", [])
        if isinstance(steps, list):
            for st in steps:
                if isinstance(st, dict):
                    if isinstance(st.get("run"), str):
                        runs.append(st["run"])
                    if isinstance(st.get("uses"), str):
                        uses_vals.append(st["uses"])
        for key in ("script", "before_script", "after_script"):
            if key in job_dict:
                collect_cmds(job_dict[key])

    if isinstance(data, dict):
        for key in ("script", "before_script", "after_script"):
            if key in data:
                collect_cmds(data[key])
        jobs = data.get("jobs", None)
        if isinstance(jobs, dict):
            for job in jobs.values():
                process_job(job)
        elif isinstance(jobs, list):
            for job in jobs:
                process_job(job)
        if "steps" in data and isinstance(data["steps"], list):
            for st in data["steps"]:
                if isinstance(st, dict):
                    if isinstance(st.get("run"), str):
                        runs.append(st["run"])
                    if isinstance(st.get("uses"), str):
                        uses_vals.append(st["uses"])
    return runs, uses_vals, lines

def find_local_action_paths(uses_vals: list[str], raw_lines: list[str]) -> list[str]:
    local_actions = []
    for u in uses_vals:
        if u.strip().startswith("./.github/actions/"):
            local_actions.append(u.strip())
    for ln in raw_lines:
        m = RE_LOCAL_ACTION_USES.search(ln)
        if m:
            local_actions.append("./.github/actions/" + m.group(1).strip())
    out, seen = [], set()
    for x in local_actions:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

def grep_patterns(text: str, patterns: list[tuple[str, re.Pattern, str]]) -> list[dict]:
    hits = []
    if not text:
        return hits
    for category, regex, confidence in patterns:
        for m in regex.finditer(text):
            s = m.start()
            line_start = text.rfind("\n", 0, s) + 1
            line_end = text.find("\n", m.end())
            if line_end == -1:
                line_end = len(text)
            snippet = text[line_start:line_end].strip()
            hits.append({"category": category, "snippet": snippet, "confidence": confidence})
    return hits

def match_supporting_files_from_runs(repo_files: list[Path], runs: list[str]) -> list[Path]:
    candidates, basenames = [], set()
    for cmd in runs:
        for m in RE_LOCAL_SCRIPT.finditer(cmd):
            basenames.add(Path(m.group("path")).name)
        if RE_MAKE.search(cmd):
            basenames.add("Makefile")
        if RE_FASTLANE.search(cmd):
            basenames.update({"Fastfile", "Appfile"})
    for p in repo_files:
        if p.name in basenames:
            candidates.append(p)
        if p.name.lower() in {"action.yml", "action.yaml"}:
            candidates.append(p)
    for p in repo_files:
        pl = p.as_posix().lower()
        if any(seg in pl for seg in ("/ci/", "/scripts/", "/tools/")) and p.suffix.lower() in {".sh", ".bash", ".cmd", ".bat"}:
            candidates.append(p)
    uniq, seen = [], set()
    for p in candidates:
        if p not in seen:
            seen.add(p); uniq.append(p)
    return uniq

def analyze_yaml_and_support(repo_id: str, yaml_paths: list[Path], all_repo_files: list[Path]) -> list[dict]:
    evidence = []
    for yml in yaml_paths:
        raw_yaml = read_text(yml)
        ytxt = strip_comments(raw_yaml)
        runs, uses_vals, raw_lines = parse_yaml_runs_uses(raw_yaml)

        # 1) scan yaml
        hits_yaml = grep_patterns(ytxt, [
            ("cloud_lab", RE_CLOUD, "high"),
            ("trigger", RE_TRIGGER, "medium"),
            ("device_setup", RE_DEVICE, "medium"),
        ])
        for h in hits_yaml:
            h.update({"repo": repo_id, "file": str(yml), "source": "yaml"})
        evidence.extend(hits_yaml)

        # 2) follow local scripts/actions/Makefile/Fastlane
        local_actions = find_local_action_paths(uses_vals, raw_lines)
        support_files = match_supporting_files_from_runs(all_repo_files, runs)

        # include action.yml inside referenced local composite actions
        for act in local_actions:
            act_base = Path(act).parts[-1].lower()
            for p in all_repo_files:
                if p.name.lower() in {"action.yml", "action.yaml"} and act_base in str(p.parent).lower():
                    support_files.append(p)

        # scan supporting files
        for sup in support_files:
            stxt = strip_comments(read_text(sup))
            hits_sup = grep_patterns(stxt, [
                ("cloud_lab", RE_CLOUD, "high"),
                ("trigger", RE_TRIGGER, "medium"),
                ("device_setup", RE_DEVICE, "medium"),
            ])
            for h in hits_sup:
                h.update({"repo": repo_id, "file": str(sup), "source": "support"})
            evidence.extend(hits_sup)
    return evidence

# ========= MAIN =========
df = pd.read_csv(MAIN_CSV, dtype=str).fillna("")

def truthy(x): return str(x).strip().lower() in {"true", "yes", "y", "1"}
def falsy(x):  return str(x).strip().lower() in {"false", "no", "n", "0", ""}

def pick_col(cands):
    cols_lower = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in cols_lower:
            return cols_lower[c.lower()]
    return None

fullcol = pick_col(["full_name", "repo", "owner_repo"])
intru_col = pick_col(["Intru_test", "instru_test", "has_androidTest"])
ci_col = pick_col(["instru_t_ci", "ci_instrumentation", "ci_instru_detected"])
if not fullcol or not intru_col or not ci_col:
    raise ValueError("Could not locate required columns (full_name, Intru_test, instru_t_ci).")

targets = df[df[intru_col].apply(truthy) & df[ci_col].apply(falsy)].copy()
targets[fullcol] = targets[fullcol].astype(str).str.strip().str.lower()

rows = []
for repo_id in targets[fullcol].unique():
    repo_files = list_repo_files(repo_id)

    if not repo_files:
        rows.append({
            "repo": repo_id,
            "status": "no_files_found",
            "evidence_json": "[]",
            "yaml_files_scanned": 0,
            "total_repo_files_scanned": 0,
            "Search_Method_Name": Search_Method_Name,  # NEW
        })
        continue

    yaml_paths = [p for p in repo_files if yaml_like(p)]
    if not yaml_paths:
        rows.append({
            "repo": repo_id,
            "status": "no_yaml_found",
            "evidence_json": "[]",
            "yaml_files_scanned": 0,
            "total_repo_files_scanned": len(repo_files),
            "Search_Method_Name": Search_Method_Name,  # NEW
        })
        continue

    ev = analyze_yaml_and_support(repo_id, yaml_paths, repo_files)

    has_cloud   = any(e["category"] == "cloud_lab" for e in ev)
    has_trigger = any(e["category"] == "trigger" for e in ev)
    has_device  = any(e["category"] == "device_setup" for e in ev)

    if has_cloud:
        status = "exists_ci_high"
    elif has_trigger and has_device:
        status = "exists_ci_high"
    elif has_trigger:
        status = "exists_ci_medium"
    else:
        status = "not_found"

    rows.append({
        "repo": repo_id,
        "status": status,
        "evidence_json": json.dumps(ev, ensure_ascii=False),
        "yaml_files_scanned": len(yaml_paths),
        "total_repo_files_scanned": len(repo_files),
        "Search_Method_Name": Search_Method_Name,  # NEW
    })

out_df = pd.DataFrame(rows).sort_values(["status", "repo"]).reset_index(drop=True)
out_df.to_csv(OUT_CSV, index=False)
print(f"Saved evidence summary -> {OUT_CSV}")
print(out_df["status"].value_counts(dropna=False))


Saved evidence summary -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.3.1_Repo_CI_Instru_Test_Scan.csv
status
not_found           1395
exists_ci_medium       9
exists_ci_high         6
Name: count, dtype: int64


create a v2.1 version which
- Search like v1.0
- correct category
- download the needed support file for fallback

First Part: fetch the missing filles

Second Part: do the analysis

In [ ]:
# -*- coding: utf-8 -*-
"""
Fetch missing fallback support files via GitHub API (multi-token, filtered).

Filters repos to those with:
  Instru_test == true  AND  instru_t_ci == false
…based on 3.2_Total_Repo.csv.

Then reads missing_support per repo from v2.1 evidence CSV and fetches those files.

Env:
  All_Tokens.env with GITHUB_TOKEN_1..6

Outputs:
  C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Support_Fallback\<owner.repo>\*
  + a fetch log CSV in the same folder.
"""

import os, time, base64, json, requests, pandas as pd
from urllib.parse import quote_plus
from pathlib import Path

# ---------- CONFIG ----------
BASE_ANALYSIS_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
TOTAL_REPO_CSV   = os.path.join(BASE_ANALYSIS_DIR, "3.2_Total_Repo.csv")                  # filter source
EVIDENCE_CSV     = os.path.join(BASE_ANALYSIS_DIR, "3.3.1_Repo_CI_Instru_Test_Scan.csv")  # has missing_support
OUT_BASE         = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Support_Fallback"
TOKENS_ENV       = r"C:\Android Mobile App\All_Tokens.env"  # contains GITHUB_TOKEN_1..6

os.makedirs(OUT_BASE, exist_ok=True)

# ---------- LOAD TOKENS & ROTATOR ----------
def load_tokens(env_path: str) -> list[str]:
    tokens = []
    if Path(env_path).exists():
        for line in Path(env_path).read_text(encoding="utf-8", errors="ignore").splitlines():
            line = line.strip()
            if not line or line.startswith("#"): continue
            if "=" in line:
                k, v = line.split("=", 1)
                if k.strip().upper().startswith("GITHUB_TOKEN_"):
                    v = v.strip().strip('"').strip("'")
                    if v: tokens.append(v)
    # Final fallback to process env
    for i in range(1, 7):
        v = os.getenv(f"GITHUB_TOKEN_{i}", "").strip()
        if v: tokens.append(v)
    # de-dupe while preserving order
    seen, uniq = set(), []
    for t in tokens:
        if t not in seen:
            seen.add(t); uniq.append(t)
    return uniq

TOKENS = load_tokens(TOKENS_ENV)
if not TOKENS:
    raise SystemExit("No GitHub tokens found in All_Tokens.env or environment (GITHUB_TOKEN_1..6).")

class GHClient:
    def __init__(self, tokens: list[str]):
        self.tokens = tokens
        self.idx = 0
        self.session = requests.Session()
        self._apply_token()

    def _apply_token(self):
        tok = self.tokens[self.idx]
        self.session.headers.update({
            "Authorization": f"Bearer {tok}",
            "Accept": "application/vnd.github+json",
            "User-Agent": "instru-fallback-fetcher"
        })

    def rotate(self):
        self.idx = (self.idx + 1) % len(self.tokens)
        self._apply_token()

    def get(self, url, params=None, retries=3, base_sleep=2):
        for attempt in range(retries):
            r = self.session.get(url, params=params, timeout=30)
            # Soft rate limit handling
            if r.status_code == 403 and "rate limit" in r.text.lower():
                # Try rotating token immediately
                self.rotate()
                time.sleep(base_sleep)
                continue
            if r.status_code in (502, 503, 504):
                time.sleep(base_sleep * (attempt + 1))
                continue
            r.raise_for_status()
            return r
        r.raise_for_status()

gh = GHClient(TOKENS)

# ---------- API HELPERS ----------
def search_code(owner, repo, filename, per_page=10):
    q = f"filename:{filename} repo:{owner}/{repo}"
    r = gh.get("https://api.github.com/search/code", params={"q": q, "per_page": per_page})
    return r.json().get("items", [])

def download_file(owner, repo, path, ref=None):
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{quote_plus(path)}"
    params = {"ref": ref} if ref else None
    r = gh.get(url, params=params)
    data = r.json()
    if isinstance(data, dict) and data.get("encoding") == "base64":
        return base64.b64decode(data["content"]), data.get("path"), data.get("sha")
    return None, None, None

# ---------- FILTER & MERGE INPUTS ----------
def truthy(x): return str(x).strip().lower() in {"true","yes","y","1"}
def falsy(x):  return str(x).strip().lower() in {"false","no","n","0",""}

total = pd.read_csv(TOTAL_REPO_CSV, dtype=str).fillna("")
# Flexible column pickup
def pick(df, cands):
    m = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in m: return m[c.lower()]
    return None

fullcol   = pick(total, ["full_name","repo","owner_repo"])
intru_col = pick(total, ["Intru_test","instru_test","has_androidTest"])
ci_col    = pick(total, ["instru_t_ci","ci_instrumentation","ci_instru_detected"])
if not fullcol or not intru_col or not ci_col:
    raise SystemExit("3.2_Total_Repo.csv must have columns: full_name, Intru_test, instru_t_ci (or equivalents).")

filter_set = (
    total[ total[intru_col].apply(truthy) & total[ci_col].apply(falsy) ][fullcol]
    .astype(str).str.strip().str.lower()
    .unique().tolist()
)
filter_set = set(filter_set)

ev = pd.read_csv(EVIDENCE_CSV, dtype=str).fillna("")
if "repo" not in ev.columns or "missing_support" not in ev.columns:
    raise SystemExit("Evidence CSV must contain columns: repo, missing_support")

# Join on repo/full_name (lower)
ev["repo_l"] = ev["repo"].astype(str).str.strip().str.lower()
ev = ev[ ev["repo_l"].isin(filter_set) & (ev["missing_support"].str.strip() != "") ]

# ---------- CORE ----------
def ensure_repo_folder(full_name: str) -> Path:
    safe = full_name.replace("/", ".")
    out_dir = Path(OUT_BASE) / safe
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

def fetch_missing_for_repo(full_name: str, missing_names: list[str]) -> list[dict]:
    owner, repo = full_name.split("/", 1)
    out_dir = ensure_repo_folder(full_name)
    results = []

    for fname in sorted({n.strip() for n in missing_names if n.strip()}):
        try:
            items = search_code(owner, repo, fname)
            if not items:
                results.append({"full_name": full_name, "filename": fname,
                                "status": "not_found", "saved_path": ""})
                continue

            def rank(item):
                p = (item.get("path") or "").lower()
                score = 0
                for pref in ("/.github/actions/", "/.github/workflows/", "/ci/", "/scripts/", "/tools/"):
                    if pref in "/" + p: score += 2
                if p.endswith("/" + fname.lower()): score += 1
                return score

            best = max(items, key=rank)
            code_path = best.get("path")
            # Prefer contents without ref, then with sha as fallback
            blob, saved_rel, sha = download_file(owner, repo, code_path, ref=None)
            if blob is None:
                blob, saved_rel, sha = download_file(owner, repo, code_path, ref=best.get("sha"))

            if blob is None:
                results.append({"full_name": full_name, "filename": fname,
                                "status": "download_failed", "saved_path": code_path or ""})
                continue

            local_name = os.path.basename(saved_rel or fname)
            local_path = out_dir / local_name
            local_path.write_bytes(blob)

            results.append({"full_name": full_name, "filename": fname,
                            "status": "saved", "saved_path": str(local_path)})
        except requests.HTTPError as e:
            results.append({"full_name": full_name, "filename": fname,
                            "status": f"http_error_{e.response.status_code}", "saved_path": ""})
        except Exception as e:
            results.append({"full_name": full_name, "filename": fname,
                            "status": f"error_{type(e).__name__}", "saved_path": ""})
    return results

def main():
    if ev.empty:
        print("No repos require fallback fetching (after filter & missing_support check).")
        return

    rows = []
    # Group evidence by normalized full_name
    for repo_l, grp in ev.groupby("repo_l"):
        # Recover canonical "owner/repo" (may exist in either CSV; prefer 3.2)
        # Build mapping from lower->canonical
        canon = next((r for r in total[fullcol].astype(str) if r.strip().lower()==repo_l), repo_l)
        missing = []
        for _, r in grp.iterrows():
            missing += [x.strip() for x in (r["missing_support"] or "").split(";") if x.strip()]
        if not missing:
            continue
        rows.extend(fetch_missing_for_repo(canon, missing))

    if rows:
        out_log = Path(OUT_BASE) / "fallback_fetch_log.csv"
        pd.DataFrame(rows).to_csv(out_log, index=False)
        print(f"Saved log -> {out_log}")
    else:
        print("Nothing fetched.")

if __name__ == "__main__":
    main()


Saved evidence summary -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.3.1_Repo_CI_Instru_Test_Scan.csv
status
not_found    1410
Name: count, dtype: int64


Part 2 - does the analysis V2.1

In [12]:
# v2.1 — Grouped detection with fallback highlighting + dual-root search (YAML-first)
# -*- coding: utf-8 -*-
import os
import re
import json
import pandas as pd
from pathlib import Path
from typing import List, Pattern, Tuple

try:
    import yaml  # pip install pyyaml
except ImportError:
    raise SystemExit("Please install pyyaml: pip install pyyaml")

# ========= CONFIG =========
BASE_ANALYSIS_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
MAIN_CSV = os.path.join(BASE_ANALYSIS_DIR, "3.2_Total_Repo.csv")  # filter source
ALL_CFG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
SUPPORT_FALLBACK_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Support_Fallback"
OUT_DIR = BASE_ANALYSIS_DIR
OUT_CSV = os.path.join(OUT_DIR, "3.3.1_Repo_CI_Instru_Test_Scan.csv")
os.makedirs(OUT_DIR, exist_ok=True)

CI_YAML_EXTS = {".yml", ".yaml"}
Search_Method_Name = "Phase 2 - Workflow & Script Trace Scan"

# ========= HELPERS =========
def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def unique_preserve(seq: List[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

COMMENT_LINE_RE = re.compile(r"(?m)^\s*(#|//|REM\b|::).*$")
def strip_comments(text: str) -> str:
    # Full-line comments removed; inline comments are kept
    return COMMENT_LINE_RE.sub("", text or "")

def read_text(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        try:
            return path.read_text(encoding="latin1", errors="ignore")
        except Exception:
            return ""

def yaml_like(path: Path) -> bool:
    n = path.name.lower()
    return (
        path.suffix.lower() in CI_YAML_EXTS
        or n in {
            ".travis.yml",
            "jenkinsfile",
            "azure-pipelines.yml",
            "bitrise.yml",
            ".gitlab-ci.yml",
            "config.yml",  # e.g., .circleci/config.yml after extraction
        }
    )

def list_repo_files(repo_id: str) -> list[Path]:
    """
    Search in BOTH All_Config_Files and Support_Fallback for this repo.
    Matches:
      - Files starting with '<owner.repo>__'
      - Files under a directory whose name starts with '<owner.repo>__'
      - Files under a folder exactly named '<owner.repo>' (used by Support_Fallback fetcher)
    """
    rid = str(repo_id).strip().lower()
    rid_prefix = rid + "__"
    results: list[Path] = []
    roots = [Path(ALL_CFG_DIR), Path(SUPPORT_FALLBACK_DIR)]

    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if not p.is_file():
                continue
            parts_lower = [part.lower() for part in p.parts]
            name_ok = p.name.lower().startswith(rid_prefix)
            parent_ok = any(part.startswith(rid_prefix) for part in parts_lower[-3:])
            folder_ok = any(part == rid for part in parts_lower[-3:])  # <owner.repo>\file
            if name_ok or parent_ok or folder_ok:
                results.append(p)
    return results

def parse_yaml_runs_uses(yaml_text: str):
    lines = yaml_text.splitlines()
    runs, uses_vals = [], []
    try:
        data = yaml.safe_load(yaml_text)
    except Exception:
        data = None

    def collect_cmds(val):
        if isinstance(val, list):
            for x in val:
                if isinstance(x, str):
                    runs.append(x)
        elif isinstance(val, str):
            runs.append(val)

    def process_job(job_dict):
        if not isinstance(job_dict, dict):
            return
        steps = job_dict.get("steps", [])
        if isinstance(steps, list):
            for st in steps:
                if isinstance(st, dict):
                    if isinstance(st.get("run"), str):
                        runs.append(st["run"])
                    if isinstance(st.get("uses"), str):
                        uses_vals.append(st["uses"])
        for key in ("script", "before_script", "after_script"):
            if key in job_dict:
                collect_cmds(job_dict[key])

    if isinstance(data, dict):
        for key in ("script", "before_script", "after_script"):
            if key in data:
                collect_cmds(data[key])
        jobs = data.get("jobs", None)
        if isinstance(jobs, dict):
            for job in jobs.values():
                process_job(job)
        elif isinstance(jobs, list):
            for job in jobs:
                process_job(job)
        if "steps" in data and isinstance(data["steps"], list):
            for st in data["steps"]:
                if isinstance(st, dict):
                    if isinstance(st.get("run"), str):
                        runs.append(st["run"])
                    if isinstance(st.get("uses"), str):
                        uses_vals.append(st["uses"])
    return runs, uses_vals, lines

# ========= GROUPED PATTERNS =========
DEVICE_SOURCES: List[Tuple[str, str, List[str]]] = [
    # Real devices via ADB
    ("Real_Device", "adb devices",      [r'(?m)^\s*(?:sudo\s+)?adb\s+devices\b', r'\badb\s+devices\b']),
    ("Real_Device", "adb get-state",    [r'(?m)^\s*(?:sudo\s+)?adb\s+get-state\b', r'\badb\s+get-state\b']),
    ("Real_Device", "adb get-serialno", [r'(?m)^\s*(?:sudo\s+)?adb\s+get-serialno\b', r'\badb\s+get-serialno\b']),
    ("Real_Device", "adb -s <serial>",  [r'(?m)^\s*(?:sudo\s+)?adb\s+-s\s+\S+\b', r'\badb\s+-s\s+\S+\b']),
    ("Real_Device", "adb install",      [r'(?m)^\s*(?:sudo\s+)?adb\s+install(\s+-r)?\b', r'\badb\s+install\b']),
    ("Real_Device", "adb shell",        [r'(?m)^\s*(?:sudo\s+)?adb\s+shell\b', r'\badb\s+shell\b']),
    ("Real_Device", "adb root",         [r'(?m)^\s*(?:sudo\s+)?adb\s+root\b', r'\badb\s+root\b']),
    ("Real_Device", "adb settings",     [r'(?m)^\s*(?:sudo\s+)?adb\s+shell\s+settings\b', r'\badb\s+shell\s+settings\b']),
    ("Real_Device", "adb input",        [r'(?m)^\s*(?:sudo\s+)?adb\s+shell\s+input\b', r'\badb\s+shell\s+input\b']),
    ("Real_Device", "adb pm grant",     [r'(?m)^\s*(?:sudo\s+)?adb\s+shell\s+pm\s+grant\b', r'\badb\s+shell\s+pm\s+grant\b']),

    # Emulator / AVD / GHA actions
    ("Emulator", "sdkmanager/avdmanager",      [r'(?m)^\s*(sdkmanager|avdmanager)\b', r'\b(sdkmanager|avdmanager)\b']),
    ("Emulator", "explicit system-image",      [r'sdkmanager\s+[\'"]system-images;android-\d+;[^\'"]+[\'"]']),
    ("Emulator", "emulator -avd/@",            [r'(?m)^\s*(?:sudo\s+)?emulator\s+(-avd|@)\S+', r'\bemulator\s+(-avd|@)\S+']),
    ("Emulator", "android-wait-for-emulator",  [r'(?m)^\s*android-wait-for-emulator\b', r'\bandroid-wait-for-emulator\b']),
    ("Emulator", "start-emulator.sh",          [r'(?m)^\s*start-emulator\.sh\b', r'\bstart-emulator\.sh\b']),
    ("Emulator", "circle-android wait-for-boot",[r'(?m)^\s*circle-android\s+wait-for-boot\b', r'\bcircle-android\s+wait-for-boot\b']),
    ("Emulator", "reactivecircus runner",      [r'uses:\s*reactivecircus/android-emulator-runner', r'reactivecircus/android-emulator-runner']),
    ("Emulator", "hannesa2 runner",            [r'uses:\s*hannesa2/action-android/emulator-run-cmd', r'hannesa2/action-android/emulator-run-cmd']),
    ("Emulator", "actions/setup-android",      [r'uses:\s*actions/setup-android', r'actions/setup-android']),
    ("Emulator", "pierotofy/setup-android",    [r'uses:\s*pierotofy/setup-android', r'pierotofy/setup-android']),
    ("Emulator", "api-level",                  [r'\bapi[-_ ]?level\b\s*:?\s*\d{2}', r'\bapi[-_ ]?level\s*\d{2}\b']),
    ("Emulator", "abi/arch",                   [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ("Emulator", "target image",               [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ("Emulator", "sdk target string",          [r'system-images;android-\d+;google_apis(?:_playstore)?;\w+']),
    ("Emulator", "ANDROID_API_LEVEL var",      [r'(?m)^\s*ANDROID_API_LEVEL\s*=\s*\d+\b', r'\bANDROID_API_LEVEL\s*=\s*\d+\b']),

    # Gradle Managed Devices
    ("GMD", "managedDevices DSL",              [r'\bmanageddevices?\b']),
    ("GMD", "ManagedVirtualDevice DSL",        [r'\bmanagedvirtualdevice\b|\bcom\.android\.build\.api\.dsl\.ManagedVirtualDevice\b']),
    ("GMD", "GMD task mentions",               [r'\bmanageddevice\w*androidtest\b']),
    ("GMD", "GHA gradle arguments/tasks",      [
        r'(?m)^\s*arguments\s*:\s*[:\w-]*manageddevice\w*androidtest\b',
        r'(?m)^\s*tasks?\s*:\s*[:\w-]*manageddevice\w*androidtest\b',
        r'\barguments\s*:\s*[:\w-]*manageddevice\w*androidtest\b',
        r'\btasks?\s*:\s*[:\w-]*manageddevice\w*androidtest\b'
    ]),

    # Third-party device labs (env/setup hints)
    ("Third_Party_Lab", "gcloud firebase",     [r'(?m)^\s*(?:sudo\s+)?gcloud(\s+beta)?(\s+--quiet)?\s+firebase\s+test\s+android\s+run\b',
                                                r'\bgcloud(\s+beta)?(\s+--quiet)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",            [r'(?m)^\s*(?:sudo\s+)?saucectl(\s+run)?\b', r'\bsaucectl(\s+run)?\b']),
    ("Third_Party_Lab", "browserstack/bstack", [r'\b(browserstack|bstack)\b']),
    ("Third_Party_Lab", "appcenter test",      [r'(?m)^\s*(?:sudo\s+)?appcenter\s+test\s+run\s+android\b', r'\bappcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "maestro cloud",       [r'(?m)^\s*maestro\s+cloud\b', r'\bmaestro\s+cloud\b']),
    ("Third_Party_Lab", "test_matrix/firebase.json", [r'\btest_matrix\.json\b|\bfirebase\.json\b']),
]

TRIGGER_SOURCES: List[Tuple[str, str, List[str]]] = [
    # Gradle triggers (anchored and relaxed)
    ("Gradle", "connectedAndroidTest",   [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+(:[\w-]+:)?connectedandroidtest\b',
                                          r'(?:\./|\.\\)?gradlew(?:\.bat)?\s+(:[\w-]+:)?connectedandroidtest\b']),
    ("Gradle", "connected.*Android.*",   [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+(:[\w-]+:)?connected.*android.*test\b',
                                          r'(?:\./|\.\\)?gradlew(?:\.bat)?\s+(:[\w-]+:)?connected.*android.*test\b']),
    ("Gradle", "connectedCheck",         [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+(:[\w-]+:)?connectedcheck\b',
                                          r'(?:\./|\.\\)?gradlew(?:\.bat)?\s+(:[\w-]+:)?connectedcheck\b']),
    ("Gradle", "createInstrCoverage",    [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*createinstrumentationtestcoveragereport\b',
                                          r'(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*createinstrumentationtestcoveragereport\b']),
    ("Gradle", "runInstrumentationTests",[r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*runinstrumentationtests\b',
                                          r'(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*runinstrumentationtests\b']),
    ("Gradle", "executeScreenshotTests", [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*executescreenshottests\b',
                                          r'(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*executescreenshottests\b']),
    ("Gradle", "orchestrator task",      [r'(?m)^\s*(?:sudo\s+)?(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*orchestrator\b',
                                          r'(?:\./|\.\\)?gradlew(?:\.bat)?\s+.*orchestrator\b']),
    ("Gradle", "yaml script -> gradle",  [r'\bscript\s*:\s*(?:\./|\.\\)?gradlew(?:\.bat)?\s+(:[\w-]+:)?connected.*']),

    # ADB / direct instrumentation
    ("ADB",    "am instrument",          [r'(?m)^\s*(?:sudo\s+)?(?:adb\s+shell\s+)?am\s+instrument\b',
                                          r'\b(?:adb\s+shell\s+)?am\s+instrument\b']),

    # Flutter triggers (restored)
    ("Flutter","flutter integration_test",[r'(?m)^\s*flutter\s+(test|drive)\b.*\bintegration_test\b',
                                           r'\bflutter\s+(test|drive)\b.*\bintegration_test\b']),

    # Third-party labs as triggers too
    ("Third_Party_Lab", "gcloud firebase",[r'(?m)^\s*(?:sudo\s+)?gcloud(?:\s+beta)?(?:\s+--quiet)?\s+firebase\s+test\s+android\s+run\b',
                                           r'\bgcloud(?:\s+beta)?(?:\s+--quiet)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",      [r'(?m)^\s*(?:sudo\s+)?saucectl(\s+run)?\b', r'\bsaucectl(\s+run)?\b']),
    ("Third_Party_Lab", "appcenter run", [r'(?m)^\s*(?:sudo\s+)?appcenter\s+test\s+run\s+android\b', r'\bappcenter\s+test\s+run\s+android\b']),
]

GROUP_CONFIDENCE = {
    "Third_Party_Lab": "high",
    "Gradle": "medium",
    "ADB": "medium",
    "Emulator": "medium",
    "Real_Device": "medium",
    "GMD": "medium",
    "Flutter": "medium",
    "Missing_Support": "low",
}

# Optional: exclude trivial gradle build/check/test from triggering
RE_EXCLUDE_GENERIC = re.compile(r'\bgradle(?:w|\.bat)?\s+.*\b(build|check|test)\b', re.I)

# Precompile grouped patterns
DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]
TRIGGER_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES]

def grep_grouped(text: str,
                 grouped_patterns: List[Tuple[str, str, List[Pattern]]],
                 default_conf: str = "medium") -> List[dict]:
    hits = []
    if not text:
        return hits
    for grp, lbl, pats in grouped_patterns:
        for p in pats:
            for m in p.finditer(text):
                s = m.start()
                line_start = text.rfind("\n", 0, s) + 1
                line_end = text.find("\n", m.end())
                if line_end == -1:
                    line_end = len(text)
                snippet = text[line_start:line_end].strip()
                hits.append({
                    "category": grp,
                    "label": lbl,
                    "snippet": snippet,
                    "confidence": GROUP_CONFIDENCE.get(grp, default_conf),
                })
    return hits

def parse_yaml_runs_uses_safe(raw_yaml: str):
    runs, uses_vals, raw_lines = parse_yaml_runs_uses(raw_yaml)
    return runs, uses_vals, raw_lines

RE_LOCAL_SCRIPT = re.compile(r"(^|\s)(?P<cmd>(?:bash|sh)\s+)?(?P<path>\.?/?(?:ci|scripts|tools|\.github/actions)/[^\s;|&]+)", re.IGNORECASE)
RE_MAKE = re.compile(r"\bmake\s+(?P<target>[\w:-]+)", re.IGNORECASE)
RE_FASTLANE = re.compile(r"\bfastlane\s+(?P<lane>[\w:_-]+)", re.IGNORECASE)

def match_supporting_files_from_runs(repo_files: list[Path], runs: list[str]) -> tuple[list[Path], list[str]]:
    """
    Return (existing_support_files, missing_referenced_filenames).
    We match by basename; fetch step will search exact names in the repo via GitHub API.
    """
    candidates, basenames = [], set()
    referenced_names = set()

    for cmd in runs:
        cmd = cmd or ""
        for m in RE_LOCAL_SCRIPT.finditer(cmd):
            pth = m.group("path")
            referenced_names.add(Path(pth).name)
            basenames.add(Path(pth).name)
        if RE_MAKE.search(cmd):
            basenames.add("Makefile"); referenced_names.add("Makefile")
        if RE_FASTLANE.search(cmd):
            basenames.update({"Fastfile", "Appfile"})
            referenced_names.update({"Fastfile", "Appfile"})

    for p in repo_files:
        if p.name in basenames or p.name.lower() in {"action.yml", "action.yaml"}:
            candidates.append(p)

    for p in repo_files:
        pl = p.as_posix().lower()
        if any(seg in pl for seg in ("/ci/", "/scripts/", "/tools/")) and p.suffix.lower() in {".sh", ".bash", ".cmd", ".bat"}:
            candidates.append(p)

    uniq, seen = [], set()
    for p in candidates:
        if p not in seen:
            seen.add(p); uniq.append(p)

    have_names = {p.name for p in uniq}
    missing = sorted(n for n in referenced_names if n not in have_names)
    return uniq, missing

def analyze_yaml_and_support(repo_id: str, yaml_paths: list[Path], all_repo_files: list[Path]) -> list[dict]:
    evidence = []
    missing_refs_all: list[str] = []

    for yml in yaml_paths:
        raw_yaml = read_text(yml)
        ytxt = strip_comments(raw_yaml)

        runs, uses_vals, raw_lines = parse_yaml_runs_uses_safe(raw_yaml)
        runs_text = "\n".join(x for x in runs if isinstance(x, str))

        # 1) Scan YAML text + normalized runs text
        hits_yaml = []
        hits_yaml += grep_grouped(ytxt, DEVICE_PATTERNS)
        hits_yaml += grep_grouped(ytxt, TRIGGER_PATTERNS)
        hits_yaml += grep_grouped(runs_text, DEVICE_PATTERNS)
        hits_yaml += grep_grouped(runs_text, TRIGGER_PATTERNS)
        for h in hits_yaml:
            h.update({"repo": repo_id, "file": str(yml), "source": "yaml"})
        evidence.extend(hits_yaml)

        # 2) Follow to support files
        support_files, missing_refs = match_supporting_files_from_runs(all_repo_files, runs)
        missing_refs_all.extend(missing_refs)

        # include action.yml inside referenced local composite actions
        # (best-effort: scanned by matcher via .github/actions, often already found)

        for sup in support_files:
            stxt = strip_comments(read_text(sup))
            hits_sup = []
            hits_sup += grep_grouped(stxt, DEVICE_PATTERNS)
            hits_sup += grep_grouped(stxt, TRIGGER_PATTERNS)
            for h in hits_sup:
                h.update({"repo": repo_id, "file": str(sup), "source": "support"})
            evidence.extend(hits_sup)

    # Record missing refs as low-confidence evidence items
    for mname in sorted(set(missing_refs_all)):
        evidence.append({
            "repo": repo_id,
            "file": "",
            "source": "support",
            "category": "Missing_Support",
            "label": "referenced_not_downloaded",
            "snippet": mname,
            "confidence": GROUP_CONFIDENCE.get("Missing_Support", "low"),
        })
    return evidence

# ========= MAIN =========
df = pd.read_csv(MAIN_CSV, dtype=str).fillna("")

def truthy(x): return str(x).strip().lower() in {"true", "yes", "y", "1"}
def falsy(x):  return str(x).strip().lower() in {"false", "no", "n", "0", ""}

def pick_col(cands):
    cols_lower = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in cols_lower:
            return cols_lower[c.lower()]
    return None

fullcol = pick_col(["full_name", "repo", "owner_repo"])
intru_col = pick_col(["Intru_test", "instru_test", "has_androidTest"])
ci_col = pick_col(["instru_t_ci", "ci_instrumentation", "ci_instru_detected"])
if not fullcol or not intru_col or not ci_col:
    raise ValueError("Could not locate required columns (full_name, Intru_test, instru_t_ci).")

# Filter: ONLY repos where Intru_test == true AND instru_t_ci == false
targets = df[df[intru_col].apply(truthy) & df[ci_col].apply(falsy)].copy()
targets[fullcol] = targets[fullcol].astype(str).str.strip().str.lower()

rows = []
for repo_id in targets[fullcol].unique():
    repo_files = list_repo_files(repo_id)

    if not repo_files:
        rows.append({
            "repo": repo_id,
            "status": "no_files_found",
            "evidence_json": "[]",
            "yaml_files_scanned": 0,
            "total_repo_files_scanned": 0,
            "Search_Method_Name": Search_Method_Name,
            "category": "",
            "fall_back": "no",
            "yaml_hit": "no",
            "support_hit": "no",
            "missing_support": "",
            "needs_refetch": "no",
        })
        continue

    yaml_paths = [p for p in repo_files if yaml_like(p)]
    if not yaml_paths:
        rows.append({
            "repo": repo_id,
            "status": "no_yaml_found",
            "evidence_json": "[]",
            "yaml_files_scanned": 0,
            "total_repo_files_scanned": len(repo_files),
            "Search_Method_Name": Search_Method_Name,
            "category": "",
            "fall_back": "no",
            "yaml_hit": "no",
            "support_hit": "no",
            "missing_support": "",
            "needs_refetch": "no",
        })
        continue

    ev = analyze_yaml_and_support(repo_id, yaml_paths, repo_files)

    # Aggregate categories + fallback + missing refs
    cats = unique_preserve([e.get("category", "") for e in ev if isinstance(e, dict) and e.get("category")])
    yaml_hit = any(isinstance(e, dict) and e.get("source") == "yaml" for e in ev)
    support_hit = any(isinstance(e, dict) and e.get("source") == "support" and e.get("category") != "Missing_Support" for e in ev)
    fall_back = "yes" if (not yaml_hit and support_hit) else "no"

    missing_list = sorted({e.get("snippet") for e in ev if e.get("category") == "Missing_Support" and e.get("snippet")})
    needs_refetch = "yes" if missing_list else "no"

    # Status (aligned with v1.0 semantics)
    has_lab     = any(c == "Third_Party_Lab" for c in cats)
    has_trigger = any(c in {"Gradle", "ADB", "Third_Party_Lab", "Flutter"} for c in cats)
    has_device  = any(c in {"Emulator", "Real_Device", "GMD"} for c in cats)

    if has_lab:
        status = "exists_ci_high"
    elif has_trigger and has_device:
        status = "exists_ci_high"
    elif has_trigger:
        status = "exists_ci_medium"
    else:
        status = "not_found"

    rows.append({
        "repo": repo_id,
        "status": status,
        "evidence_json": json.dumps(ev, ensure_ascii=False),
        "yaml_files_scanned": len(yaml_paths),
        "total_repo_files_scanned": len(repo_files),
        "Search_Method_Name": Search_Method_Name,
        "category": ", ".join(cats),
        "fall_back": fall_back,
        "yaml_hit": "yes" if yaml_hit else "no",
        "support_hit": "yes" if support_hit else "no",
        "missing_support": "; ".join(missing_list),
        "needs_refetch": needs_refetch,
    })

out_df = pd.DataFrame(rows).sort_values(["status", "repo"]).reset_index(drop=True)
out_df.to_csv(OUT_CSV, index=False)
print(f"Saved evidence summary -> {OUT_CSV}")
print(out_df["status"].value_counts(dropna=False))


Saved evidence summary -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.3.1_Repo_CI_Instru_Test_Scan.csv
status
not_found           1398
exists_ci_high         6
exists_ci_medium       6
Name: count, dtype: int64
